In [1]:
import numpy as np
import seaborn as sns
import pandas as pd
from matplotlib import pyplot as plt
plt.rcParams['font.family']='Times New Roman,Microsoft YaHei'# 设置字体族，中文为微软雅黑，英文为Times New Roman
plt.rcParams['mathtext.fontset'] = 'stix' # 设置数%matplotlib qt学公式字体为stix
plt.style.use('seaborn-v0_8-paper')
# 设置全局参数
plt.rcParams['figure.facecolor'] = 'white'  # 设置图形的背景为透明
plt.rcParams['axes.facecolor'] = 'white'    # 设置轴域的背景为透明
plt.rcParams['savefig.facecolor'] = 'white' # 保存图像时背景透明
import matplotlib
# matplotlib.use('TkAgg')
%matplotlib inline

In [16]:
# dfmusic = pd.read_pickle("df_chordkey,pkl")
dfbird = pd.read_pickle('final_birds_df.pkl')
dfclasse = pd.read_pickle('final_classes_df.pkl')
dftempo=pd.read_pickle("music_var_tempo.pkl")

In [17]:
# 去除重复列
dftempo = dftempo.loc[:, ~dftempo.columns.duplicated()]

# 定义高度替换规则
def replace_level(row):
    if row['place'] == 'ZS':
        return '3 m' if row['level'] == 1 else f"{row['level']}m"
    elif row['place'] == 'JH':
        return {1: '1.5 m', 2: '4 m', 3: '8 m', 4: '14 m'}.get(row['level'], f"{row['level']}m")
    elif row['place'] == 'CM':
        return {1: '1.5 m', 2: '10 m', 3: '16 m', 4: '22 m'}.get(row['level'], f"{row['level']}m")
    return f"{row['level']}m"

# 替换 level 列的值
dftempo['level'] = dftempo.apply(replace_level, axis=1)

In [18]:
import ast
def get_highest_tempo(tempo_estimates_str):
    try:
        # 将字符串转换为集合
        tempo_estimates_str = tempo_estimates_str.replace('{', '[').replace('}', ']')  # 将集合形式转换为列表形式
        tempos = ast.literal_eval(tempo_estimates_str)
        
        if isinstance(tempos, list):
            # 转换为 NumPy 数组
            tempos_array = np.array(tempos)
            
            # 提取权重列
            weights = tempos_array[:, 1].astype(float)
            # 提取节奏列
            tempos_values = tempos_array[:, 0].astype(float)
            
            # 找到权重最高的节奏
            highest_index = np.argmax(weights)
            highest_tempo = tempos_values[highest_index]
            return highest_tempo
        else:
            print(f"Error: Expected a list, but got a different type: {type(tempos)}")
            return None
    except (ValueError, SyntaxError, TypeError) as e:
        # 处理解析错误或格式错误
        print(f"Error processing {tempo_estimates_str}: {e}")
        return None
    
    
# 应用 get_highest_tempo 函数
dftempo['tempo'] = dftempo['Tempo_Estimates'].apply(get_highest_tempo)

# 删除原始的 Tempo_Estimates 列
dftempo = dftempo.drop(columns=['Tempo_Estimates'])

In [32]:
df = pd.merge(dfclasse ,dftempo,on=['Date', 'place', 'level'], how='inner')

In [33]:
# 分类映射
category_mapping = {
    'Bird': ['Bird', 'Bird vocalization, bird call, bird song', 'Chirp, tweet', 'Duck', 'Fowl'],
    'Poultry': ['Duck', 'Fowl'],
    'Mammals': ['Rodents, rats, mice'],
    'Wild Animals': ['Frog','Animal', 'Wild animals'],
    'Environmental Sounds': ['White noise', 'Wind noise (microphone)','Rain', 'Rain on surface', 'Water', 'Wind'],
    'Insect': ['Insect'],
    'Transportation': ['Vehicle', 'Motorboat, speedboat', 'Boat, Water vehicle'],
    'Human Speech': ['Speech'],
    'Music and Instruments': ['Music', 'Synthesizer'],
    'Machinery and Tools': ['Camera', 'Single-lens reflex camera'],
}


# 将所有类别映射为 keyword -> category
keyword_to_category = {}
for category, keywords in category_mapping.items():
    for keyword in keywords:
        keyword_to_category[keyword] = category

# 优化后的分类函数
def classify_sound_class(sound_class):
    for keyword, category in keyword_to_category.items():
        if keyword in sound_class:
            return category
    return 'Other'  # 未匹配到的类别归为 'Other'


# 假设 df 是你的数据框
# 生成新列并进行分类
df['class'] = df['class'].apply(classify_sound_class)
# 按照 category_mapping 的顺序创建分类类型
category_order = list(category_mapping.keys())
# 将 'class_category' 列设置为分类类型，并指定排序顺序
df['class'] = pd.Categorical(df['class'], categories=category_order, ordered=True)
df = df.sort_values(by='class')

In [34]:
# 转换字符串到 NumPy 数组
df['Onsets'] = df['Onsets'].apply(lambda x: np.fromstring(x.strip('{}'), sep=','))

In [35]:
df[['Date','level','place','tempo','class']].to_csv("BPMvocieclass.csv")
df.to_pickle("BPMvocieclass.pkl")

In [4]:
df=pd.read_pickle("BPMvocieclass.pkl")
df.to_csv("BPMvocieclassoneset.csv")

In [5]:
df=pd.read_pickle("BPMbirdclass.pkl")
df.to_csv("BPMvociebirdoneset.csv")

In [23]:
df

,Date,place,level,class,Onsets,tempo
316046,2023-03-16 08:00:00,JH,4 m,Bird,"[0.54, 3.28, 3.42, 6.25, 8.83, 10.46, 11.92, 1...",181.818182
477662,2023-04-27 09:30:00,ZS,3 m,Bird,"[0.71, 5.08, 9.38, 11.48, 15.07, 21.25, 21.59,...",68.965517
1066120,2024-02-15 17:30:00,ZS,3 m,Bird,"[0.95, 1.39, 7.31, 35.17, 41.48, 42.85, 49.15,...",181.818182
1066119,2024-02-15 17:30:00,ZS,3 m,Bird,"[0.95, 1.39, 7.31, 35.17, 41.48, 42.85, 49.15,...",181.818182
1066118,2024-02-15 17:30:00,ZS,3 m,Bird,"[0.95, 1.39, 7.31, 35.17, 41.48, 42.85, 49.15,...",181.818182
...,...,...,...,...,...,...
841078,2023-11-19 03:00:00,ZS,3 m,Machinery and Tools,[],44.444444
1217035,2024-04-20 20:10:00,JH,1.5 m,Machinery and Tools,[7.52],153.846154
1217034,2024-04-20 20:10:00,JH,1.5 m,Machinery and Tools,[7.52],153.846154
841089,2023-11-19 03:20:00,CM,16 m,Machinery and Tools,[],109.090909


In [25]:
df = pd.merge(dfbird ,dftempo,on=['Date', 'place', 'level'], how='inner')

In [26]:
# 栖息地分类映射
habitat_mapping = {
    'Aquatic and Wetland Surface Birds': ['Eurasian Wigeon', 'Tundra Swan', 'Mallard', 
                                          'Greater White-fronted Goose', 'Common Goldeneye', 
                                          'Green-winged Teal', 'Eurasian Coot', 'Eurasian Moorhen'],

    'Shoreline and Marsh Birds': ['Eurasian Curlew', 'Whimbrel'],

    'Grassland Ground Birds': ['Olive-backed Pipit', 'Yellow-browed Bunting', 'Yellow-billed Grosbeak'],

    'Shrub Layer Birds': ['Light-vented Bulbul', 'Chinese Hwamei', 'Japanese Tit', 
                          'Silver-throated Tit', 'Chinese Blackbird', 'Pale-legged Leaf Warbler'],

    'Lower Canopy Birds': ['Pale Thrush', 'Oriental Magpie'],

    'Upper Canopy and Aerial Birds': ["Swinhoe's White-eye"]
}

# 将所有物种映射到栖息地类别
species_to_habitat = {}
for habitat, species_list in habitat_mapping.items():
    for species in species_list:
        species_to_habitat[species] = habitat

# 分类函数
def classify_bird_habitat(species):
    return species_to_habitat.get(species, 'Other')  # 若未匹配到则归类为 'Other'

# 应用分类函数
df['habitat'] = df['bird'].copy().apply(classify_bird_habitat)

# 根据分类映射顺序创建分类类型
habitat_order = list(habitat_mapping.keys())
df['habitat'] = pd.Categorical(df['habitat'], categories=habitat_order, ordered=True)

In [27]:
# 转换字符串到 NumPy 数组
df['Onsets'] = df['Onsets'].apply(lambda x: np.fromstring(x.strip('{}'), sep=','))

In [31]:
df[['Date', 'level', 'place', 'tempo','bird','habitat']].to_csv("BPMbirdclass.csv")
df.to_pickle("BPMbirdclass.pkl")

In [29]:
df

,Date,place,level,bird,Onsets,tempo,habitat
0,2022-10-27 11:40:00,CM,10 m,Eurasian Curlew,"[22.31, 23.94, 34.05, 35.4, 36.71, 37.89, 40.1...",153.846154,Shoreline and Marsh Birds
1,2022-10-27 12:50:00,CM,16 m,Tundra Swan,"[0.89, 1.61, 2.65, 3.0, 3.7, 3.88, 4.42, 4.78,...",162.162162,Aquatic and Wetland Surface Birds
2,2022-10-27 13:10:00,CM,10 m,Mallard,"[3.89, 4.4, 10.71, 16.45, 31.62, 52.61, 52.9, ...",117.647059,Aquatic and Wetland Surface Birds
3,2022-10-27 13:30:00,CM,1.5 m,Light-vented Bulbul,"[4.38, 26.68, 45.88, 52.26, 53.86, 56.72, 58.26]",214.285714,Shrub Layer Birds
4,2022-10-27 13:30:00,CM,10 m,Light-vented Bulbul,"[4.33, 19.45, 19.83, 25.06, 25.62, 25.7, 25.97...",162.162162,Shrub Layer Birds
...,...,...,...,...,...,...,...
216846,2024-06-13 15:00:00,JH,8 m,Mallard,"[6.23, 6.97, 8.85, 9.2, 14.24, 26.72, 26.9, 27...",200.000000,Aquatic and Wetland Surface Birds
216847,2024-06-13 15:00:00,JH,8 m,Common Goldeneye,"[6.23, 6.97, 8.85, 9.2, 14.24, 26.72, 26.9, 27...",200.000000,Aquatic and Wetland Surface Birds
216848,2024-06-13 15:10:00,JH,8 m,Eurasian Wigeon,"[5.73, 6.59, 15.74, 15.87, 16.23, 17.78, 19.37...",222.222222,Aquatic and Wetland Surface Birds
216849,2024-06-13 15:10:00,JH,8 m,Mallard,"[5.73, 6.59, 15.74, 15.87, 16.23, 17.78, 19.37...",222.222222,Aquatic and Wetland Surface Birds


In [11]:
# # 转换字符串到 NumPy 数组
# df['Onsets'] = df['Onsets'].apply(lambda x: np.fromstring(x.strip('{}'), sep=','))

In [12]:
# # 计算全局平均趋势，首先确定所有数组长度一致
# min_length = min(len(x) for x in df['Onsets'])
# df['Onsets'] = df['Onsets'].apply(lambda x: x[:min_length])
# global_mean = df['Onsets'].apply(pd.Series).mean()

In [13]:

# # 创建一个 FacetGrid，根据 place 和 month 分类
# g = sns.FacetGrid(df, col="Month", row="place", hue="level", palette="viridis", 
#                   margin_titles=True, height=4, aspect=2)
# 
# # 绘制每个 level 的 200 个随机样本
# def plot_samples(data, color, label, **kwargs):
#     # 随机选择 200 个样本，并降低透明度
#     sample_data = data.sample(n=200, random_state=42)
#     for _, row in sample_data.iterrows():
#         plt.plot(row['Onsets'], color=color, alpha=0.3, linewidth=0.5)
# 
# # 映射上述函数到每个子图
# g.map_dataframe(plot_samples)
# 
# # 在每个子图中绘制总平均趋势
# g.map(plt.plot, global_mean, color="red", linewidth=2, label="Global Mean")
# 
# # 添加图例和美化图表
# g.add_legend(title="Level")
# g.set_axis_labels("Index", "Onsets Value")
# g.set_titles("{col_name} | {row_name}")
# 
# plt.tight_layout()
# plt.show()